In [ ]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

import shap

# ==============================
# LOAD DATA (STABLE)
# ==============================

df = pd.read_csv("data/raw/train.csv")

# ==============================
# FEATURE ENGINEERING
# ==============================

df["TotalSF"] = df["GrLivArea"] + df["TotalBsmtSF"]
df["HouseAge"] = df["YrSold"] - df["YearBuilt"]
df["QualityScore"] = df["OverallQual"] * df["OverallCond"]

# ==============================
# FEATURES
# ==============================

features = [
    "GrLivArea",
    "OverallQual",
    "GarageCars",
    "TotalBsmtSF",
    "YearBuilt",
    "TotalSF",
    "HouseAge",
    "QualityScore"
]

target = "SalePrice"

df_model = df[features + [target]].dropna()

X = df_model[features]
y = df_model[target]

# ==============================
# TRAIN TEST SPLIT
# ==============================

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

# ==============================
# MODEL
# ==============================

model = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

model.fit(X_train, y_train)

# ==============================
# FEATURE IMPORTANCE
# ==============================

importances = model.feature_importances_

plt.figure(figsize=(10,5))
plt.barh(features, importances)
plt.title("Feature Importance (Random Forest)")
plt.show()

# ==============================
# SHAP EXPLANATION
# ==============================

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

# global importance
shap.summary_plot(shap_values, X_test)

# JS init (pour notebooks)
shap.initjs()

# explanation d'une ligne
i = 0

shap.force_plot(
    explainer.expected_value,
    shap_values[i],
    X_test.iloc[i]
)

ValueError: Invalid dataset handle: house-prices-advanced-regression-techniques